# Projeto Final M3 – Classificação de Imagens  
## Dataset D: DeepSat SAT-6

**Disciplina:** Processamento de Imagens  
**Curso:** Ciência da Computação  
**Tema:** Classificação automática de imagens de satélite usando CNN  
**Autores:** coloque aqui o nome dos integrantes do grupo

---

## Objetivo do projeto

Desenvolver uma solução em Python para classificar automaticamente imagens do dataset **DeepSat SAT-6** em uma das seis classes de cobertura do solo:

1. Barren Land  
2. Trees  
3. Grassland  
4. Roads  
5. Buildings  
6. Water Bodies  

O projeto inclui:

- Download automático do dataset pelo Kaggle;
- Pré-processamento das imagens;
- Data augmentation;
- Treinamento de uma CNN;
- Avaliação por acurácia, precisão, recall, F1-score e matriz de confusão;
- Exibição de exemplos de inferência;
- Interface simples para upload e classificação de uma imagem.

> Observação: o SAT-6 possui imagens pequenas de 28x28 pixels com 4 canais: vermelho, verde, azul e infravermelho próximo, chamado de NIR.

# 1. Instalação e importação das bibliotecas

Execute a célula abaixo.  
No Colab, ative GPU em: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.

In [ ]:
!pip install -q kaggle opencv-python scikit-learn matplotlib pillow

In [ ]:
import os
import io
import random
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

# 2. Download do dataset DeepSat SAT-6 direto do Kaggle

Este notebook baixa o dataset direto do Kaggle, sem precisar colocar no Google Drive.

## Como gerar o arquivo `kaggle.json`

1. Entre em sua conta no Kaggle.
2. Vá em **Account / Settings**.
3. Na seção **API**, clique em **Create New Token**.
4. Será baixado o arquivo `kaggle.json`.
5. Ao executar a próxima célula, envie esse arquivo quando solicitado.

O dataset usado é: `crawford/deepsat-sat6`.

In [ ]:
DATASET_SLUG = "crawford/deepsat-sat6"
DATA_DIR = "/content/deepsat-sat6"

def configurar_kaggle():
    kaggle_dir = os.path.expanduser("~/.kaggle")
    kaggle_json_destino = os.path.join(kaggle_dir, "kaggle.json")

    if os.path.exists(kaggle_json_destino):
        print("Arquivo kaggle.json já configurado.")
        return

    os.makedirs(kaggle_dir, exist_ok=True)

    try:
        from google.colab import files
        print("Envie o arquivo kaggle.json baixado do site do Kaggle.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Você precisa enviar o arquivo chamado kaggle.json.")
        with open(kaggle_json_destino, "wb") as f:
            f.write(uploaded["kaggle.json"])
        os.chmod(kaggle_json_destino, 0o600)
        print("kaggle.json configurado com sucesso.")
    except Exception as e:
        print("Não foi possível configurar automaticamente o kaggle.json.")
        print("Erro:", e)
        print("Coloque manualmente o arquivo em ~/.kaggle/kaggle.json.")

def baixar_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)

    arquivos_esperados = [
        "X_train_sat6.csv",
        "y_train_sat6.csv",
        "X_test_sat6.csv",
        "y_test_sat6.csv"
    ]

    if all(os.path.exists(os.path.join(DATA_DIR, arq)) for arq in arquivos_esperados):
        print("Dataset já encontrado em:", DATA_DIR)
        return

    configurar_kaggle()
    print("Baixando dataset. Isso pode demorar alguns minutos...")
    comando = [
        "kaggle", "datasets", "download",
        "-d", DATASET_SLUG,
        "-p", DATA_DIR,
        "--unzip"
    ]
    subprocess.run(comando, check=True)
    print("Download finalizado.")

baixar_dataset()

print("\nArquivos encontrados:")
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        print(os.path.join(root, file))

# 3. Carregamento dos dados

O dataset oficial possui:

- 324.000 imagens de treino;
- 81.000 imagens de teste;
- Cada imagem tem 28x28 pixels e 4 canais;
- Cada rótulo está em formato one-hot com 6 posições.

Para deixar o treinamento mais leve no Colab, o notebook começa usando uma parte do dataset:

- `TRAIN_N = 32400`
- `TEST_N = 8100`

Se quiser usar mais dados, aumente esses valores.  
Para usar o dataset inteiro, coloque `TRAIN_N = None` e `TEST_N = None`, mas isso pode consumir muita RAM.

In [ ]:
TRAIN_N = 32400   # use None para carregar todo o treino
TEST_N = 8100     # use None para carregar todo o teste

IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHANNELS = 4
NUM_CLASSES = 6

# Ordem de classes usada na descrição original do SAT-6.
CLASS_NAMES = [
    "Barren Land",
    "Trees",
    "Grassland",
    "Roads",
    "Buildings",
    "Water Bodies"
]

train_data_path = os.path.join(DATA_DIR, "X_train_sat6.csv")
train_label_path = os.path.join(DATA_DIR, "y_train_sat6.csv")
test_data_path = os.path.join(DATA_DIR, "X_test_sat6.csv")
test_label_path = os.path.join(DATA_DIR, "y_test_sat6.csv")

def carregar_csv_sat6(caminho_x, caminho_y, nrows=None):
    print(f"Lendo {os.path.basename(caminho_x)}...")
    X_df = pd.read_csv(caminho_x, header=None, dtype=np.uint8, nrows=nrows)

    print(f"Lendo {os.path.basename(caminho_y)}...")
    y_df = pd.read_csv(caminho_y, header=None, dtype=np.uint8, nrows=nrows)

    X = X_df.to_numpy().reshape(-1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    y = y_df.to_numpy().astype("float32")

    del X_df, y_df
    return X, y

X_train_raw, y_train_onehot = carregar_csv_sat6(train_data_path, train_label_path, TRAIN_N)
X_test_raw, y_test_onehot = carregar_csv_sat6(test_data_path, test_label_path, TEST_N)

print("X_train_raw:", X_train_raw.shape, X_train_raw.dtype)
print("y_train_onehot:", y_train_onehot.shape, y_train_onehot.dtype)
print("X_test_raw:", X_test_raw.shape, X_test_raw.dtype)
print("y_test_onehot:", y_test_onehot.shape, y_test_onehot.dtype)

y_train_idx = np.argmax(y_train_onehot, axis=1)
y_test_idx = np.argmax(y_test_onehot, axis=1)

# 4. Visualização inicial das imagens

Como o dataset tem 4 canais, a visualização abaixo mostra apenas os canais RGB.  
O quarto canal, NIR, será usado no treinamento do modelo.

In [ ]:
def mostrar_amostras(X, y_idx, class_names, n=12):
    indices = np.random.choice(len(X), size=n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(12, 3 * rows))

    for i, idx in enumerate(indices):
        img_rgb = X[idx, :, :, :3]
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img_rgb)
        plt.title(class_names[y_idx[idx]])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

mostrar_amostras(X_train_raw, y_train_idx, CLASS_NAMES, n=12)

# 5. Pré-processamento das imagens

Técnicas aplicadas:

1. **Equalização adaptativa de histograma com CLAHE nos canais RGB**  
   Ajuda a melhorar contraste local e reduzir problemas de variação de iluminação.

2. **Normalização para o intervalo [0, 1]**  
   Facilita o treinamento da rede neural.

3. **Preservação do canal NIR**  
   O quarto canal contém informação multiespectral útil para diferenciar classes visualmente parecidas.

4. **Data augmentation durante o treinamento**  
   Aumenta a capacidade de generalização do modelo com rotações, espelhamentos e pequenos zooms.

In [ ]:
APPLY_CLAHE = True

def aplicar_clahe_rgb(batch_uint8):
    """
    Aplica CLAHE somente nos três canais RGB.
    O canal NIR é preservado sem equalização.
    """
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    saida = np.empty_like(batch_uint8)

    for i in range(batch_uint8.shape[0]):
        img = batch_uint8[i]
        for c in range(3):
            saida[i, :, :, c] = clahe.apply(img[:, :, c])
        saida[i, :, :, 3] = img[:, :, 3]

    return saida

def preprocessar_imagens(batch_uint8, aplicar_clahe=True):
    if aplicar_clahe:
        batch_uint8 = aplicar_clahe_rgb(batch_uint8)
    batch_float = batch_uint8.astype("float32") / 255.0
    return batch_float

X_train_pp = preprocessar_imagens(X_train_raw, aplicar_clahe=APPLY_CLAHE)
X_test_pp = preprocessar_imagens(X_test_raw, aplicar_clahe=APPLY_CLAHE)

print("Intervalo treino:", X_train_pp.min(), X_train_pp.max())
print("Intervalo teste:", X_test_pp.min(), X_test_pp.max())
print("Shape:", X_train_pp.shape)

In [ ]:
# Comparação visual antes e depois do pré-processamento
idx = np.random.randint(0, len(X_train_raw))

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(X_train_raw[idx, :, :, :3])
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(X_train_pp[idx, :, :, :3])
plt.title("Pré-processada RGB")
plt.axis("off")

plt.tight_layout()
plt.show()

# 6. Separação entre treino e validação

O dataset já possui divisão oficial entre treino e teste.  
Aqui, apenas separamos uma parte do treino para validação durante o treinamento.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_pp,
    y_train_onehot,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_idx
)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test_pp.shape, y_test_onehot.shape)

# 7. Desenvolvimento do algoritmo de classificação

A solução usa uma **CNN treinada do zero**.

## Fluxo do método

1. Entrada: imagem 28x28x4;
2. Data augmentation;
3. Blocos convolucionais com `Conv2D`, `BatchNormalization`, `MaxPooling2D` e `Dropout`;
4. `GlobalAveragePooling2D`;
5. Camada densa;
6. Saída `Softmax` com 6 classes.

A CNN é adequada porque aprende automaticamente padrões espaciais das imagens, como textura, bordas, regiões homogêneas, vegetação, água, estradas e áreas construídas.

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(0.10, seed=SEED),
        layers.RandomZoom(0.10, seed=SEED),
    ],
    name="data_augmentation"
)

def criar_modelo_cnn():
    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name="imagem_sat6")

    x = data_augmentation(inputs)

    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classe")(x)

    model = keras.Model(inputs, outputs, name="CNN_DeepSat_SAT6")
    return model

model = criar_modelo_cnn()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 8. Treinamento

O treinamento usa:

- `EarlyStopping` para parar quando a validação não melhora;
- `ReduceLROnPlateau` para reduzir a taxa de aprendizado quando necessário.

Se a acurácia ficar abaixo do esperado, aumente `EPOCHS` ou carregue mais imagens aumentando `TRAIN_N` e `TEST_N`.

In [ ]:
BATCH_SIZE = 128
EPOCHS = 25

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
def plotar_historico(history):
    hist = history.history

    plt.figure(figsize=(8, 5))
    plt.plot(hist["accuracy"], label="Acurácia treino")
    plt.plot(hist["val_accuracy"], label="Acurácia validação")
    plt.xlabel("Época")
    plt.ylabel("Acurácia")
    plt.title("Evolução da acurácia")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="Loss treino")
    plt.plot(hist["val_loss"], label="Loss validação")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title("Evolução da função de perda")
    plt.legend()
    plt.grid(True)
    plt.show()

plotar_historico(history)

# 9. Avaliação do modelo no conjunto de teste

Nesta etapa são calculadas as métricas obrigatórias:

- Acurácia;
- Precisão;
- Recall;
- F1-score;
- Matriz de confusão.

In [ ]:
test_loss, test_acc = model.evaluate(X_test_pp, y_test_onehot, batch_size=BATCH_SIZE, verbose=0)

y_prob = model.predict(X_test_pp, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test_onehot, axis=1)

acc = accuracy_score(y_true, y_pred)

print(f"Loss de teste: {test_loss:.4f}")
print(f"Acurácia de teste pelo Keras: {test_acc:.4f}")
print(f"Acurácia de teste pelo sklearn: {acc:.4f}")
print("\nRelatório de classificação:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues", values_format="d", xticks_rotation=45)
plt.title("Matriz de Confusão - DeepSat SAT-6")
plt.tight_layout()
plt.show()

# 10. Exemplos de inferência no conjunto de teste

A célula abaixo mostra imagens do conjunto de teste com:

- Classe real;
- Classe prevista;
- Probabilidade da previsão.

In [ ]:
def mostrar_predicoes(model, X_raw, X_preprocessado, y_true_idx, class_names, n=12):
    indices = np.random.choice(len(X_preprocessado), size=n, replace=False)
    probs = model.predict(X_preprocessado[indices], verbose=0)
    preds = np.argmax(probs, axis=1)

    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(14, 3.5 * rows))

    for i, idx in enumerate(indices):
        real = class_names[y_true_idx[idx]]
        pred = class_names[preds[i]]
        conf = probs[i, preds[i]]
        correto = "OK" if preds[i] == y_true_idx[idx] else "ERRO"

        plt.subplot(rows, cols, i + 1)
        plt.imshow(X_raw[idx, :, :, :3])
        plt.title(f"{correto}\nReal: {real}\nPred: {pred} ({conf:.2%})", fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

mostrar_predicoes(model, X_test_raw, X_test_pp, y_true, CLASS_NAMES, n=12)

# 11. Salvamento do modelo

O modelo treinado será salvo para uso posterior.

In [ ]:
MODEL_PATH = "/content/modelo_deepsat_sat6.keras"
model.save(MODEL_PATH)
print("Modelo salvo em:", MODEL_PATH)

# 12. Aplicação simples para inferência por upload

Como imagens de satélite com canal NIR não são capturadas por webcam comum, esta aplicação permite fazer upload de uma imagem externa.

A função aceita:

- PNG/JPG RGB comum: o canal NIR é aproximado pela média dos canais RGB;
- Imagem 28x28 ou qualquer tamanho: ela será redimensionada para 28x28;
- Para melhor demonstração, use imagens exportadas do próprio conjunto de teste.

A próxima célula cria alguns exemplos RGB do conjunto de teste na pasta `/content/exemplos_teste`.

In [ ]:
EXEMPLOS_DIR = "/content/exemplos_teste"
os.makedirs(EXEMPLOS_DIR, exist_ok=True)

indices = np.random.choice(len(X_test_raw), size=20, replace=False)

for i, idx in enumerate(indices):
    classe_real = CLASS_NAMES[y_true[idx]].replace(" ", "_")
    img_rgb = Image.fromarray(X_test_raw[idx, :, :, :3])
    caminho = os.path.join(EXEMPLOS_DIR, f"exemplo_{i:02d}_real_{classe_real}.png")
    img_rgb.save(caminho)

print("Exemplos salvos em:", EXEMPLOS_DIR)
print("Arquivos:")
for arq in os.listdir(EXEMPLOS_DIR)[:10]:
    print(os.path.join(EXEMPLOS_DIR, arq))

In [ ]:
def preparar_imagem_upload(imagem_pil):
    """
    Prepara uma imagem enviada pelo usuário:
    - converte para RGB;
    - redimensiona para 28x28;
    - cria um quarto canal aproximado para NIR;
    - aplica o mesmo pré-processamento usado no treino.
    """
    img_rgb = imagem_pil.convert("RGB").resize((IMG_WIDTH, IMG_HEIGHT))
    arr_rgb = np.array(img_rgb, dtype=np.uint8)

    # Como PNG/JPG comum não tem NIR, usamos a média dos canais RGB como aproximação.
    nir_aproximado = np.mean(arr_rgb, axis=2, keepdims=True).astype(np.uint8)
    arr_4c = np.concatenate([arr_rgb, nir_aproximado], axis=2)

    arr_4c = arr_4c.reshape(1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    arr_pp = preprocessar_imagens(arr_4c, aplicar_clahe=APPLY_CLAHE)
    return arr_rgb, arr_pp

def classificar_imagem_pil(model, imagem_pil):
    img_rgb, img_pp = preparar_imagem_upload(imagem_pil)
    prob = model.predict(img_pp, verbose=0)[0]
    pred_idx = int(np.argmax(prob))

    plt.figure(figsize=(4, 4))
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.title(f"Classe prevista: {CLASS_NAMES[pred_idx]}\nConfiança: {prob[pred_idx]:.2%}")
    plt.show()

    print("Probabilidades por classe:")
    for nome, p in zip(CLASS_NAMES, prob):
        print(f"{nome:15s}: {p:.4f}")

    return CLASS_NAMES[pred_idx], prob[pred_idx]

In [ ]:
# Interface de upload para Google Colab
try:
    from google.colab import files

    print("Faça upload de uma imagem PNG/JPG para classificação.")
    uploaded = files.upload()

    for nome_arquivo, conteudo in uploaded.items():
        print("\nArquivo:", nome_arquivo)
        imagem = Image.open(io.BytesIO(conteudo))
        classe, confianca = classificar_imagem_pil(model, imagem)
        print(f"Resultado final: {classe} com confiança de {confianca:.2%}")

except Exception as e:
    print("Esta célula foi feita para rodar no Google Colab.")
    print("Erro:", e)

# 13. Análise e discussão dos resultados

Após executar o notebook, complete a análise com os valores obtidos.

## Pontos para comentar na apresentação

- O modelo usa uma CNN treinada do zero para aprender padrões espaciais das imagens de satélite.
- O pré-processamento normaliza os dados e melhora contraste dos canais RGB com CLAHE.
- O canal NIR foi mantido porque ajuda a diferenciar vegetação, água e outros tipos de cobertura do solo.
- O data augmentation reduz overfitting, pois simula pequenas variações de orientação e escala.
- A matriz de confusão permite observar quais classes foram mais confundidas pelo modelo.
- Caso ocorra confusão entre classes, isso pode ser explicado pela semelhança visual entre algumas categorias e pela baixa resolução das imagens, que possuem apenas 28x28 pixels.
- Para melhorar o resultado, seria possível usar mais dados, treinar por mais épocas ou testar arquiteturas como ResNet, MobileNet ou EfficientNet adaptadas para imagens pequenas.

## Texto-base para conclusão

Neste trabalho foi desenvolvida uma solução de classificação automática de imagens de sensoriamento remoto utilizando o dataset DeepSat SAT-6. A abordagem utilizou pré-processamento, normalização, data augmentation e uma rede neural convolucional treinada do zero. O desempenho foi avaliado por meio de acurácia, precisão, recall, F1-score e matriz de confusão. Os resultados mostram que a CNN é capaz de aprender padrões relevantes nas imagens, classificando diferentes tipos de cobertura do solo. As principais dificuldades estão relacionadas à semelhança visual entre algumas classes e à baixa resolução das imagens.

# 14. Referências

- Kaggle: DeepSat SAT-6 Airborne Dataset  
- Basu et al. DeepSat: A Learning Framework for Satellite Imagery  
- TensorFlow/Keras Documentation  
- Scikit-learn Documentation  
- OpenCV Documentation